In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 42.5 MB/s eta 0:00:00


In [12]:
!nvidia-smi

Tue Dec 16 13:08:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             52W /  400W |   14209MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from huggingface_hub import login

# You will be prompted to paste your HF token (make a token at: https://huggingface.co/settings/tokens)
login()


In [16]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re
import time
from typing import List, Dict, Any, Tuple

# --------- MODEL LOADING ----------
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    # A100 is best with BF16
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    # Force model fully onto GPU 0 (avoid CPU offload / sharding surprises)
    device_map = {"": 0} if torch.cuda.is_available() else None

    # Try fastest attention implementation first
    for attn_impl in ("flash_attention_2", "sdpa", None):
        try:
            print(f"Loading model (attn_implementation={attn_impl}) ...")
            kwargs = dict(
                device_map=device_map,
                torch_dtype=dtype,
            )
            if attn_impl is not None:
                kwargs["attn_implementation"] = attn_impl

            m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
            m.eval()
            return m
        except Exception as e:
            print(f"Failed loading with attn_implementation={attn_impl}: {e}")

    # Final fallback
    print("Loading model with default attention implementation...")
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map=device_map,
        torch_dtype=dtype,
    )
    m.eval()
    return m

model = load_model()
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --------- 20 PATENT-RELATED NER LABELS ----------
LABELS = [
    "INVENTION",
    "COMPONENT",
    "SUBSYSTEM",
    "MATERIAL",
    "CHEMICAL",
    "BIOMOLECULE",
    "COMPOSITION",
    "PROCESS_STEP",
    "METHOD",
    "PARAMETER",
    "MEASUREMENT",
    "CONDITION",
    "FUNCTION",
    "SIGNAL",
    "CONTROL",
    "SOFTWARE",
    "HARDWARE",
    "FIGURE_REF",
    "CLAIM_ELEMENT",
    "PRIOR_ART",
    "UNCLASSIFIED_ENTITY",
]

# --------- FASTAPI APP ----------
app = FastAPI()

class PredictRequest(BaseModel):
    data: list  # [{"text": "..."}]

def _extract_json_array(s: str) -> List[Dict[str, Any]]:
    m = re.search(r"\[\s*{.*?}\s*\]", s, flags=re.DOTALL)
    if not m:
        m = re.search(r"\[.*\]", s, flags=re.DOTALL)
    if not m:
        return []
    blob = m.group(0)
    try:
        parsed = json.loads(blob)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        return []
    return []

def _find_span(text: str, entity_text: str, start_from: int = 0) -> Tuple[int, int]:
    if not entity_text:
        return (-1, -1)
    idx = text.find(entity_text, start_from)
    if idx == -1:
        return (-1, -1)
    return (idx, idx + len(entity_text))

def extract_spans(text: str) -> List[Dict[str, Any]]:
    labels_str = ", ".join(LABELS)

    prompt = f"""
You are a patent NER engine.

Extract span entities from the input text. Use ONLY these labels:
{labels_str}

Rules:
- Return ONLY a valid JSON array.
- Each item must be an object with keys: "text" and "label".
- "text" MUST be an exact substring copied from the input (verbatim, same casing).
- "label" MUST be exactly one of the allowed labels.
- Prefer longer, specific spans over tiny fragments.
- Do NOT create overlapping spans unless unavoidable; if unsure, skip.

Input text:
\"\"\"{text}\"\"\"

Return JSON array now:
""".strip()

    # Always move inputs to the same GPU explicitly
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Generation config tuned for NER JSON
    gen_kwargs = dict(
        max_new_tokens=200,          # << lower than 512 for latency
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    t0 = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    t1 = time.time()

    # Decode only newly generated tokens (not the whole prompt)
    prompt_len = inputs["input_ids"].shape[-1]
    gen_ids = output[0][prompt_len:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Basic perf log
    new_tokens = gen_ids.shape[-1]
    dt = max(t1 - t0, 1e-6)
    print(f"[NER] {new_tokens} tokens in {dt:.2f}s -> {new_tokens/dt:.1f} tok/s")

    items = _extract_json_array(decoded)

    spans = []
    next_search_pos: Dict[str, int] = {}
    used_ranges: List[Tuple[int, int]] = []

    def overlaps(a: Tuple[int, int], b: Tuple[int, int]) -> bool:
        return not (a[1] <= b[0] or b[1] <= a[0])

    for it in items:
        if not isinstance(it, dict):
            continue
        ent_text = str(it.get("text", "")).strip()
        label = str(it.get("label", "")).strip()

        if label not in LABELS or not ent_text:
            continue

        start_from = next_search_pos.get(ent_text, 0)
        start, end = _find_span(text, ent_text, start_from=start_from)

        if start == -1:
            start, end = _find_span(text, ent_text, start_from=0)
            if start == -1:
                continue

        candidate = (start, end)

        if any(overlaps(candidate, r) for r in used_ranges):
            cursor = end
            found = False
            while True:
                s2, e2 = _find_span(text, ent_text, start_from=cursor)
                if s2 == -1:
                    break
                cand2 = (s2, e2)
                if not any(overlaps(cand2, r) for r in used_ranges):
                    start, end = s2, e2
                    candidate = cand2
                    found = True
                    break
                cursor = e2
            if not found:
                continue

        used_ranges.append(candidate)
        next_search_pos[ent_text] = end

        spans.append(
            {
                "start": start,
                "end": end,
                "text": text[start:end],
                "labels": [label],
            }
        )

    return spans

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}

@app.api_route("/setup", methods=["GET", "POST"])
def setup():
    return {
        "from_name": "ner",
        "to_name": "text",
        "type": "labels",
        "labels": LABELS,
    }

@app.post("/predict")
def predict(request: PredictRequest):
    results = []
    for item in request.data:
        text = item["text"]
        spans = extract_spans(text)

        results.append(
            {
                "result": [
                    {
                        "from_name": "ner",
                        "to_name": "text",
                        "type": "labels",
                        "value": span,
                    }
                    for span in spans
                ],
                "score": 1.0,
            }
        )
    return results


Loading tokenizer...
Loading model (attn_implementation=flash_attention_2) ...
Failed loading with attn_implementation=flash_attention_2: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.
Loading model (attn_implementation=sdpa) ...
Using device: cuda:0


In [5]:
import uvicorn, threading, nest_asyncio, time, requests

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)  # give it a few seconds to start

# quick test: local health
print(requests.get("http://127.0.0.1:8000/health").json())


INFO:     Started server process [718]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:48032 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok'}


In [6]:
from google.colab import output

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
public_url


'https://8000-gpu-a100-s-2cj2h8l4rfh22-c.us-central1-1.prod.colab.dev'

In [18]:
from pyngrok import ngrok
from dotenv import load_dotenv
load_dotenv()
import os
# kill old tunnels in this session
ngrok.kill()

# your auth token
ngrok.set_auth_token("2PgsprcdKolcczw6ru6HXbLcYfC_7cUSXnTdho7wqZyHYotoF")

# A) random domain
# public_url = ngrok.connect(addr="127.0.0.1:8000")

# B) your reserved free domain
public_url = ngrok.connect(
    addr="127.0.0.1:8000",
    domain="empiristic-mariyah-unprophetically.ngrok-free.dev"
)

print("Public URL:", public_url)


Public URL: NgrokTunnel: "https://empiristic-mariyah-unprophetically.ngrok-free.dev" -> "http://127.0.0.1:8000"


InvalidSchema: No connection adapters were found for '127.0.0.1:8000/api/predictions/bulk/'